# 02 · CTC recognition: learning without alignments

*Voice & Speech Models*, notebook 2 of 4

A recogniser hears frames and must write symbols, but nobody tells it which frames belong to which symbol. **Connectionist Temporal Classification** (CTC) solves this by adding a *blank* label and summing over every way of aligning the transcript to the frames. In this notebook you build CTC from its definition, check it against brute force, give it a gradient, plug it into your tiny tensor engine and train a recogniser that transcribes sequences of tones it has never heard.

## What you will build

- `ctc_collapse`: the rule that turns a frame-by-frame path into a transcript;
- `ctc_forward`: the CTC forward algorithm in log space, checked against brute-force enumeration of every alignment;
- `ctc_loss_and_grad`: the CTC loss and its gradient with respect to the frame log-probabilities, checked numerically and then wrapped as an operation of the tiny tensor engine;
- `greedy_decode`: best-path decoding;
- a small recogniser, trained with your CTC loss on generated tone sequences and scored by its symbol error rate on held-out utterances.

## Learning objectives

By the end of this notebook you will be able to:

1. apply the collapse rule (merge repeats, then remove blanks) and explain why a blank must separate repeated symbols;
2. build the extended target with its allowed transitions, and compute the CTC likelihood with the forward algorithm in log space;
3. derive the gradient of the CTC loss from forward and backward variables, and verify it with finite differences;
4. add a custom operation with its own backward function to an autograd engine;
5. train and evaluate a CTC recogniser, and explain three failure modes: impossible target lengths, shifted noise conditions and repeated symbols with no pause between them.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function in the cell provided and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Cells marked **🧪 Try it** invite you to change one value and run them again.

## Dependencies

- **Tier 1:** NumPy for every calculation and Matplotlib for the plots, plus the standard-library modules `itertools` (for the brute-force check) and `time`.
- The tiny tensor engine from Base Model Training and the feature functions from notebook 01 are pasted into hidden recap cells, so this notebook runs on its own.
- Nothing is downloaded: every utterance is generated from tones and noise with seeded random generators.

## Setup

Run this cell first. It imports NumPy and Matplotlib, creates a seeded random generator and defines `practice`, the helper that checks each function you type against a reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import itertools
import time

import matplotlib.pyplot as plt

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below is the reverse-mode autograd engine from Base Model Training, unchanged: `Tensor` (with `@`, broadcasting arithmetic, `relu`, `tanh`, `log_softmax`, `softmax`, indexing and `backward`), `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it has been gradient-checked. Run the cell; open it only if you want to look at the code again.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

## 📦 Recap: what you built earlier

The hidden cell below defines the feature pipeline from notebook 01 under its usual names: `make_tone`, `hann_window`, `frame_signal`, `stft_power`, `hz_to_mel`, `mel_to_hz`, `mel_filterbank`, `log_mel` and `log_mel_spectrogram`. Run it; open it only if you want to look back at the code.

In [ ]:
# 📦 Recap: the pieces from notebook 01, under their usual names.

def make_tone(frequency, duration, sample_rate, amplitude=1.0, phase=0.0):
    """A sampled sine wave of shape (int(round(duration * sample_rate)),)."""
    n_samples = int(round(duration * sample_rate))
    t = np.arange(n_samples) / sample_rate
    return amplitude * np.sin(2 * np.pi * frequency * t + phase)


def hann_window(length):
    """The periodic Hann window: 0.5 - 0.5 cos(2 pi n / length) for n = 0, ..., length - 1."""
    return 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(length) / length)


def frame_signal(x, frame_length, hop):
    """Overlapping frames of x: shape (1 + (len(x) - frame_length) // hop, frame_length)."""
    n_frames = 1 + (len(x) - frame_length) // hop
    return np.array([x[m * hop: m * hop + frame_length] for m in range(n_frames)])


def stft_power(x, frame_length, hop):
    """Power spectrogram |rfft(window * frame)|^2: shape (M, frame_length // 2 + 1)."""
    frames = frame_signal(x, frame_length, hop) * hann_window(frame_length)
    return np.abs(np.fft.rfft(frames, axis=1)) ** 2


def hz_to_mel(frequency):
    """HTK mel scale."""
    return 2595.0 * np.log10(1.0 + np.asarray(frequency, dtype=float) / 700.0)


def mel_to_hz(mel):
    """Inverse of hz_to_mel."""
    return 700.0 * (10.0 ** (np.asarray(mel, dtype=float) / 2595.0) - 1.0)


def mel_filterbank(n_mels, n_fft, sample_rate, f_min=0.0, f_max=None):
    """Triangular mel filters: shape (n_mels, n_fft // 2 + 1)."""
    f_max = sample_rate / 2 if f_max is None else f_max
    edges = mel_to_hz(np.linspace(hz_to_mel(f_min), hz_to_mel(f_max), n_mels + 2))
    freqs = np.arange(n_fft // 2 + 1) * sample_rate / n_fft
    left, centre, right = edges[:-2, None], edges[1:-1, None], edges[2:, None]
    rising = (freqs - left) / (centre - left)
    falling = (right - freqs) / (right - centre)
    return np.maximum(0.0, np.minimum(rising, falling))


def log_mel(power, filterbank, eps=1e-10):
    """Log band energies: (M, n_bins) power and (n_mels, n_bins) filters give (M, n_mels)."""
    return np.log(power @ filterbank.T + eps)


def log_mel_spectrogram(x, sample_rate, frame_length=256, hop=128, n_mels=20):
    """Waveform (N,) -> log-mel features (M, n_mels)."""
    return log_mel(stft_power(x, frame_length, hop), mel_filterbank(n_mels, frame_length, sample_rate))

## 1 · The task: transcribe a sequence of tones

Each utterance is a sequence of two to four symbols from the alphabet **a**, **b** and **c**. Each symbol is a tone (500, 850 or 1400 Hz) lasting 64 to 112 ms, and symbols are separated by 48 to 80 ms of silence, all with a little background noise. Repeats such as "aab" are allowed. The recogniser sees log-mel frames every 16 ms, so a symbol covers four to seven frames, and **it is never told where any symbol starts or ends**: it only receives the audio and the transcript.

Label index 0 is reserved for the CTC blank, written `_`; indices 1, 2 and 3 are a, b and c. Tones stand in for speech sounds here because their acoustics are easy to inspect; the alignment problem is the same one that speech recognisers solve.

In [ ]:
SAMPLE_RATE, FRAME_LENGTH, HOP, N_MELS = 8000, 256, 128, 20
BLANK = 0
ALPHABET = "_abc"                                   # index 0 is the CTC blank
TONE_HZ = {1: 500.0, 2: 850.0, 3: 1400.0}


def make_utterance(labels, rng, noise=0.05):
    """Silence, then every symbol as a tone of random length followed by a pause, plus noise."""
    pieces = [np.zeros(int(rng.integers(256, 513)))]                  # 32 to 64 ms of lead-in
    for label in labels:
        n = int(rng.integers(512, 897))                                 # a 64 to 112 ms tone
        fade = np.minimum(1.0, np.minimum(np.arange(n), n - 1 - np.arange(n)) / 40)   # 5 ms fade in and out
        tone = make_tone(TONE_HZ[label] * rng.uniform(0.97, 1.03), n / SAMPLE_RATE, SAMPLE_RATE,
                         rng.uniform(0.3, 0.9), rng.uniform(0.0, 2 * np.pi))
        pieces += [fade * tone, np.zeros(int(rng.integers(384, 641)))]  # then 48 to 80 ms of silence
    x = np.concatenate(pieces)
    return x + noise * rng.normal(size=len(x))


def to_text(labels):
    return "".join(ALPHABET[int(k)] for k in labels)


example_labels = [1, 1, 3]
example_wave = make_utterance(example_labels, np.random.default_rng(0))
example_features = log_mel_spectrogram(example_wave, SAMPLE_RATE)
print(f"transcript '{to_text(example_labels)}': {len(example_wave)} samples = {len(example_wave) / SAMPLE_RATE:.2f} s "
      f"-> {example_features.shape[0]} frames of {example_features.shape[1]} log-mel values")

fig, axes = plt.subplots(2, 1, figsize=(8, 4.2))
axes[0].plot(np.arange(len(example_wave)) / SAMPLE_RATE, example_wave, lw=0.5)
axes[0].set_title(f"The utterance '{to_text(example_labels)}': two a tones with a pause between them, then c")
axes[1].imshow(example_features.T, origin="lower", aspect="auto", cmap="magma")
axes[1].set_xlabel("frame (16 ms hop)"); axes[1].set_ylabel("mel band")
plt.tight_layout(); plt.show()

## 2 · Paths, blanks and the collapse rule

A CTC model outputs, for every frame $t$, a probability distribution over the symbols **and** the blank: an array of shape `(T, C)` with $C = 4$ here. A **path** $\pi$ picks one label per frame, so it has length $T$. The collapse map $\mathcal{B}$ turns a path into a transcript in two steps, in this order:

1. merge consecutive repeated labels;
2. remove the blanks.

So `aa_bb` becomes `ab`, `a_a` becomes `aa`, and `aaa` becomes `a`. The order matters: removing blanks first would turn `a_a` into `aa` and then merge it into a single `a`. This is why the blank exists. It lets the model write the same symbol twice in a row, and it gives the model somewhere to put frames that belong to no symbol at all, such as pauses and transitions.

In [ ]:
path = [1, 1, 0, 0, 2, 2, 0, 2]
print("path:          ", to_text(path))
print("merge repeats: ", to_text([1, 0, 2, 0, 2]))
print("remove blanks: ", to_text([1, 2, 2]))

### ✍️ Your turn: `ctc_collapse`

Write `ctc_collapse(path, blank=0)`:

- **input:** a sequence of integer labels (a list or a NumPy array) of length $T$;
- **output:** a Python list of `int` labels: the transcript.

Hint: walk through the path remembering the previous label. Append a label when it differs from the previous one **and** is not the blank. Convert with `int(label)` so that NumPy integers become plain Python integers.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def ctc_collapse_reference(path, blank=0):
    out, previous = [], None
    for label in path:
        if label != previous and label != blank:
            out.append(int(label))
        previous = label
    return out

In [ ]:
def ctc_collapse(path, blank=0):
    """Merge repeated labels, then drop blanks: [1, 1, 0, 1, 2] -> [1, 1, 2]."""
    out, previous = [], None
    # ✍️ Your code here (a loop over path, four lines)
    raise NotImplementedError

In [ ]:
ctc_collapse = practice("ctc_collapse", ctc_collapse, ctc_collapse_reference, cases=[
    ([1, 1, 0, 2, 2],),
    ([1, 0, 1],),                        # the blank keeps both a symbols
    ([0, 0, 0],),                        # all blanks: an empty transcript
    ([3, 3, 3, 1],),
    (np.array([2, 0, 0, 2, 2, 1]),),     # a NumPy array works too
])

**Expected behaviour.** `[1, 1, 0, 2, 2]` gives `[1, 2]` ("ab"), `[1, 0, 1]` gives `[1, 1]` ("aa"), an all-blank path gives `[]`, and the NumPy path gives `[2, 2, 1]` ("bba").

## 3 · Every alignment, by brute force

CTC treats the frames as independent given the audio, so a path has probability $\prod_t y_t(\pi_t)$, where $y_t(k)$ is the model's probability of label $k$ at frame $t$. The probability of a transcript $z$ adds up every path that collapses to it, and the loss is its negative logarithm:

$$P(z \mid x) = \sum_{\pi \,:\, \mathcal{B}(\pi) = z} \;\prod_{t=0}^{T-1} y_t(\pi_t), \qquad \mathcal{L} = -\ln P(z \mid x).$$

No alignment is chosen by hand: training raises the probability of all of them at once, and the model discovers which frames belong to which symbol. For tiny cases we can list every one of the $C^T$ paths. The cell below works with log-probabilities $\ln y_t(k)$, so a product becomes a sum and `np.logaddexp(a, b)` $= \ln(e^a + e^b)$ adds probabilities without leaving log space.

In [ ]:
def log_softmax_rows(z):
    """Row-wise log-softmax in NumPy: turns any (T, C) scores into log-probabilities."""
    shifted = z - z.max(axis=1, keepdims=True)
    return shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))


def brute_force_log_prob(log_probs, target, blank=0):
    """log P(target) by trying every one of the C ** T paths. Only for tiny cases."""
    T, C = log_probs.shape
    total = -np.inf
    for path in itertools.product(range(C), repeat=T):
        if ctc_collapse(path, blank) == list(target):
            total = np.logaddexp(total, sum(log_probs[t, label] for t, label in enumerate(path)))
    return float(total)


tiny = log_softmax_rows(np.random.default_rng(1).normal(size=(4, 2)))   # 4 frames, labels {blank, a}
valid = [path for path in itertools.product(range(2), repeat=4) if ctc_collapse(path) == [1, 1]]
print(f"{len(valid)} of the {2 ** 4} paths over 4 frames collapse to 'aa':", [to_text(p) for p in valid])
print("log P('aa') by brute force:", round(brute_force_log_prob(tiny, [1, 1]), 4))

**What to notice.** Every valid path for "aa" has a blank somewhere between the two a symbols. A path such as `aaaa` collapses to a single "a" and is not counted.

## 4 · The forward algorithm in log space

Brute force grows like $C^T$: for 100 frames and 30 labels it would take longer than the age of the universe. The **forward algorithm** shares the work with dynamic programming. First insert a blank around and between the symbols. This **extended target** $\ell'$ has length $S = 2L + 1$; for $z =$ "ab" it is `_a_b_`. A valid path walks through $\ell'$ from left to right, one position per frame. At each new frame it may

- **stay** on the same position,
- **step** to the next position, or
- **skip** the blank between two symbols, but only if those two symbols differ: skipping the blank in `a_a` would merge the two a symbols.

It must start at position 0 or 1 and finish at position $S - 1$ or $S - 2$. The **forward variable** $\alpha_t(s)$ is the total probability of every path prefix that covers frames $0, \dots, t$ and ends at position $s$:

$$\alpha_0(0) = y_0(\ell'_0), \qquad \alpha_0(1) = y_0(\ell'_1), \qquad \alpha_0(s) = 0 \text{ for } s \ge 2,$$

$$\alpha_t(s) = y_t(\ell'_s)\,\Big[\alpha_{t-1}(s) + \alpha_{t-1}(s-1) + [\text{skip allowed at } s]\;\alpha_{t-1}(s-2)\Big],$$

where a skip into position $s$ is allowed when $\ell'_s$ is not the blank and $\ell'_s \ne \ell'_{s-2}$. Finally $P(z \mid x) = \alpha_{T-1}(S-1) + \alpha_{T-1}(S-2)$.

A product of hundreds of probabilities underflows to 0 in floating point, so we store $\ln \alpha$ instead: multiplication becomes addition, the bracket becomes nested `np.logaddexp` calls, and "probability 0" is $-\infty$. Conveniently, `np.logaddexp(-np.inf, -np.inf)` is $-\infty$ without any warning. Shapes: `log_probs` is `(T, C)`, the target has length $L$, and `log_alpha` is `(T, S)`.

In [ ]:
def extend_with_blanks(target, blank=0):
    """[a, b] -> [_, a, _, b, _]: the extended target of length 2 * len(target) + 1."""
    extended = np.full(2 * len(target) + 1, blank, dtype=int)
    extended[1::2] = target
    return extended


ext = extend_with_blanks([1, 1, 2])
can_skip = np.zeros(len(ext), dtype=bool)
can_skip[2:] = (ext[2:] != BLANK) & (ext[2:] != ext[:-2])
print("extended target:        ", " ".join(to_text(ext)))
print("skip allowed into each: ", " ".join(str(int(v)) for v in can_skip))

### ✍️ Your turn: `ctc_forward`

Write `ctc_forward(log_probs, target, blank=0)` that returns `log_alpha` of shape `(T, S)`:

1. `ext = extend_with_blanks(target, blank)`, `T = log_probs.shape[0]`, `S = len(ext)`, and `log_alpha = np.full((T, S), -np.inf)`;
2. set the first frame: `log_alpha[0, 0] = log_probs[0, ext[0]]` and, if `S > 1`, `log_alpha[0, 1] = log_probs[0, ext[1]]`;
3. compute `can_skip` as in the demo;
4. for `t` from 1 to `T - 1`: build `stay = log_alpha[t - 1]`; `step`, a copy shifted right by one with $-\infty$ in front; and `skip`, shifted right by two, with $-\infty$ wherever a skip is not allowed. Then set `log_alpha[t] = np.logaddexp(np.logaddexp(stay, step), skip) + log_probs[t, ext]`.

Hints: `step = np.full(S, -np.inf)` followed by `step[1:] = log_alpha[t - 1, :-1]` makes the shift without touching the row it reads from. `log_probs[t, ext]` picks, for every position, the log-probability of that position's label.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def ctc_forward_reference(log_probs, target, blank=0):
    ext = extend_with_blanks(target, blank)
    T, S = log_probs.shape[0], len(ext)
    log_alpha = np.full((T, S), -np.inf)
    log_alpha[0, 0] = log_probs[0, ext[0]]
    if S > 1:
        log_alpha[0, 1] = log_probs[0, ext[1]]
    can_skip = np.zeros(S, dtype=bool)
    can_skip[2:] = (ext[2:] != blank) & (ext[2:] != ext[:-2])
    for t in range(1, T):
        stay = log_alpha[t - 1]
        step = np.full(S, -np.inf)
        step[1:] = log_alpha[t - 1, :-1]
        skip = np.full(S, -np.inf)
        skip[2:] = log_alpha[t - 1, :-2]
        skip[~can_skip] = -np.inf
        log_alpha[t] = np.logaddexp(np.logaddexp(stay, step), skip) + log_probs[t, ext]
    return log_alpha

In [ ]:
def ctc_forward(log_probs, target, blank=0):
    """Log forward variables log alpha, shape (T, 2 * len(target) + 1)."""
    # ✍️ Your code here (about 15 lines: set-up, the first frame, can_skip, then the loop over t)
    raise NotImplementedError

In [ ]:
forward_rng = np.random.default_rng(2)
ctc_forward = practice("ctc_forward", ctc_forward, ctc_forward_reference, cases=[
    (log_softmax_rows(forward_rng.normal(size=(5, 3))), [1, 2]),
    (log_softmax_rows(forward_rng.normal(size=(4, 3))), [1, 1]),
    (log_softmax_rows(forward_rng.normal(size=(3, 3))), []),       # an empty transcript: only blanks
    (log_softmax_rows(forward_rng.normal(size=(2, 3))), [1, 1]),   # too few frames: the end stays at -inf
])

The loss reads the last frame at the two allowed end positions. The next cell compares it with brute force on six tiny cases, including repeated symbols and an empty transcript.

In [ ]:
def ctc_loss(log_probs, target, blank=0):
    """-log P(target | frames): the last frame, at the final blank or the final symbol."""
    log_alpha = ctc_forward(log_probs, target, blank)
    return float(-np.logaddexp.reduce(log_alpha[-1, -2:]))


check_rng = np.random.default_rng(7)
print(" T  target   forward    brute force   paths")
for T, target in [(5, [1, 2]), (5, [1, 1]), (4, [2]), (6, [1, 2, 1]), (3, []), (6, [2, 2, 2])]:
    log_probs = log_softmax_rows(check_rng.normal(size=(T, 3)))
    forward_value = -ctc_loss(log_probs, target)
    brute_value = brute_force_log_prob(log_probs, target)
    count = sum(ctc_collapse(p) == target for p in itertools.product(range(3), repeat=T))
    print(f"{T:2d}  {to_text(target) or '(empty)':7s}  {forward_value:9.5f}  {brute_value:12.5f}  {count:6d}")
    assert np.isclose(forward_value, brute_value), "the forward algorithm should match brute force"

print("\nloss for 'aa' in only 2 frames:", ctc_loss(log_softmax_rows(check_rng.normal(size=(2, 3))), [1, 1]))

**Expected behaviour.** Every row agrees to five decimal places, although the forward algorithm visits each `(frame, position)` pair once instead of listing hundreds of paths. The last line prints `inf`: "aa" needs at least three frames (a, blank, a), so with two frames no path exists and $P = 0$. Real CTC losses report this as an infinite loss; check target lengths before training (Exercise 1 counts the minimum number of frames).

## 5 · The gradient of the CTC loss

To train, we need $\partial \mathcal{L} / \partial y_t(k)$, where from now on $y_t(k)$ means the **log**-probability that the model outputs (the result of `log_softmax`). Define the **backward variable** $\beta_t(s)$ as the total probability of every path *suffix* that is at position $s$ at frame $t$ and finishes correctly, counting the frames *after* $t$ only. Then $\alpha_t(s)\,\beta_t(s)$ is the total probability of the valid paths that pass through position $s$ at frame $t$, and since every valid path is at exactly one position at frame $t$,

$$P = \sum_{s} \alpha_t(s)\,\beta_t(s) \quad \text{for every } t .$$

A path's probability contains the factor $e^{y_t(\ell'_s)}$ exactly once, so differentiating with respect to $y_t(k)$ keeps exactly the paths whose label at frame $t$ is $k$:

$$\frac{\partial P}{\partial y_t(k)} = \sum_{s \,:\, \ell'_s = k} \alpha_t(s)\,\beta_t(s), \qquad \frac{\partial \mathcal{L}}{\partial y_t(k)} = -\frac{1}{P} \sum_{s \,:\, \ell'_s = k} \alpha_t(s)\,\beta_t(s) = -\gamma_t(k).$$

$\gamma_t(k)$ is the probability, given the audio and the transcript, that frame $t$ is aligned to label $k$. For every frame the $\gamma_t(k)$ add up to 1, and they are zero for labels that do not occur in the transcript.

**The backward variables come for free.** Reverse the frames and the target and run your forward algorithm: flipped back, its result $\hat\beta_t(s)$ is the probability of every suffix from position $s$ at frame $t$ *including* frame $t$'s own factor. This works because reversing the target reverses the extended target, and the stay, step and skip rules read the same in both directions. Removing that one factor gives

$$\ln \alpha_t(s) + \ln \beta_t(s) = \ln \alpha_t(s) + \ln \hat\beta_t(s) - y_t(\ell'_s) .$$

The array of these sums, which we call the **occupancy**, has shape `(T, S)`; the gradient has shape `(T, C)`.

In [ ]:
target = np.array([1, 2])
log_probs = log_softmax_rows(np.random.default_rng(3).normal(size=(5, 3)))
log_beta_hat = ctc_forward(log_probs[::-1], target[::-1])[::-1, ::-1]
occupancy = ctc_forward(log_probs, target) + log_beta_hat - log_probs[:, extend_with_blanks(target)]
print("log P from each frame:", np.round(np.logaddexp.reduce(occupancy, axis=1), 6))
print("ctc_loss gives        ", round(-ctc_loss(log_probs, target), 6))

Every frame gives the same $\ln P$, which confirms the identity $P = \sum_s \alpha_t(s) \beta_t(s)$.

### ✍️ Your turn: `ctc_loss_and_grad`

Write `ctc_loss_and_grad(log_probs, target, blank=0)` that returns a tuple `(loss, grad)`:

1. `target = np.asarray(target, dtype=int)`, `ext = extend_with_blanks(target, blank)`, then `log_alpha` from your `ctc_forward` and `log_beta_hat` from `ctc_forward` on the reversed frames and target, flipped back with `[::-1, ::-1]`;
2. `log_p = np.logaddexp.reduce(log_alpha[-1, -2:])`. If it is not finite, the target cannot fit: return `(float("inf"), np.zeros_like(log_probs))`, because there is no gradient to follow;
3. `occupancy = log_alpha + log_beta_hat - log_probs[:, ext]`;
4. start from `grad = np.zeros_like(log_probs)` and, for every label `k` in `np.unique(ext)`, set `grad[:, k] = -np.exp(np.logaddexp.reduce(occupancy[:, ext == k], axis=1) - log_p)`;
5. return `(float(-log_p), grad)`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def ctc_loss_and_grad_reference(log_probs, target, blank=0):
    target = np.asarray(target, dtype=int)
    ext = extend_with_blanks(target, blank)
    log_alpha = ctc_forward(log_probs, target, blank)
    log_beta_hat = ctc_forward(log_probs[::-1], target[::-1], blank)[::-1, ::-1]
    log_p = np.logaddexp.reduce(log_alpha[-1, -2:])
    grad = np.zeros_like(log_probs)
    if not np.isfinite(log_p):
        return float("inf"), grad
    occupancy = log_alpha + log_beta_hat - log_probs[:, ext]
    for k in np.unique(ext):
        grad[:, k] = -np.exp(np.logaddexp.reduce(occupancy[:, ext == k], axis=1) - log_p)
    return float(-log_p), grad

In [ ]:
def ctc_loss_and_grad(log_probs, target, blank=0):
    """(-log P(target), d loss / d log_probs of shape (T, C)); (inf, zeros) if the target cannot fit."""
    # ✍️ Your code here (about 12 lines, following the five steps)
    raise NotImplementedError

In [ ]:
grad_rng = np.random.default_rng(4)
ctc_loss_and_grad = practice("ctc_loss_and_grad", ctc_loss_and_grad, ctc_loss_and_grad_reference, cases=[
    (log_softmax_rows(grad_rng.normal(size=(6, 4))), [1, 2]),
    (log_softmax_rows(grad_rng.normal(size=(5, 4))), [3, 3]),
    (log_softmax_rows(grad_rng.normal(size=(4, 4))), []),
    (log_softmax_rows(grad_rng.normal(size=(2, 4))), [1, 1]),   # impossible: (inf, zeros)
])

Now check the formula against finite differences. The derivation never assumed that the $y_t(k)$ are normalised, so the check perturbs each entry of an arbitrary score array on its own, using `numerical_grad` and `relative_error` from the engine.

In [ ]:
scores = Tensor(np.random.default_rng(8).normal(size=(6, 4)))
check_target = [1, 2, 2]
loss_value, analytic = ctc_loss_and_grad(scores.data, check_target)
numerical = numerical_grad(lambda: Tensor(ctc_loss(scores.data, check_target)), scores)
gradient_error = relative_error(analytic, numerical)
print(f"loss on raw scores {loss_value:.4f} (negative because these scores are not normalised log-probabilities)")
print(f"relative error between your gradient and finite differences: {gradient_error:.2e}")
print("every row of -grad adds up to 1:", np.allclose(-analytic.sum(axis=1), 1.0))
assert gradient_error < 1e-6

plt.figure(figsize=(6, 2.6))
plt.imshow(-analytic.T, origin="lower", aspect="auto", cmap="Blues", vmin=0, vmax=1)
plt.yticks(range(4), list(ALPHABET)); plt.xlabel("frame"); plt.colorbar(label="occupancy")
plt.title("Where the transcript 'abb' is aligned: occupancy gamma per frame and label")
plt.tight_layout(); plt.show()

## 6 · CTC as an operation of the tiny tensor engine

Every `Tensor` in the engine remembers its parents and a `backward` function that maps the gradient arriving at its output to one gradient per parent. A new operation only needs those two things. The CTC loss is a scalar, so the gradient arriving at it is a scalar `g` (1 when it is the final loss, $1/B$ when it is averaged over a batch of $B$ utterances), and the gradient it passes back to `log_probs` is `g * grad`, using the gradient you just derived.

In [ ]:
def ctc_op(log_probs, target, blank=0):
    """CTC loss as an engine operation: value -log P(target), gradient from your ctc_loss_and_grad."""
    loss, grad = ctc_loss_and_grad(log_probs.data, target, blank)
    return Tensor(loss, requires_grad=log_probs.requires_grad, parents=(log_probs,),
                  backward=lambda g: (g * grad,), op="ctc")


# Check 1: the operation on its own.
y = Tensor(np.random.default_rng(9).normal(size=(7, 4)), requires_grad=True)
y.grad = None
ctc_op(y, [1, 3, 3]).backward()
error_alone = relative_error(y.grad, numerical_grad(lambda: ctc_op(y, [1, 3, 3]), y))

# Check 2: composed with the engine's log_softmax, as it will be in the model.
logits = Tensor(np.random.default_rng(10).normal(size=(7, 4)), requires_grad=True)
build = lambda: ctc_op(logits.log_softmax(axis=-1), [1, 3, 3])
logits.grad = None
build().backward()
error_composed = relative_error(logits.grad, numerical_grad(build, logits))
print(f"relative error, op alone: {error_alone:.2e}; through log_softmax: {error_composed:.2e}")
assert error_alone < 1e-6 and error_composed < 1e-6

softmax_probs = np.exp(logits.log_softmax(axis=-1).data)
_, grad_wrt_log_probs = ctc_loss_and_grad(logits.log_softmax(axis=-1).data, [1, 3, 3])
print("gradient with respect to the logits = softmax - occupancy:", np.allclose(logits.grad, softmax_probs + grad_wrt_log_probs))

**What to notice.** Through `log_softmax`, the gradient with respect to the logits is $\text{softmax}_t(k) - \gamma_t(k)$: "what the model predicts minus where the alignment says the frame belongs", the same shape of formula as ordinary cross-entropy with a soft, computed target.

## 7 · Greedy decoding

At test time there is no transcript. **Greedy** (best-path) decoding takes the most likely label in every frame and collapses the result. This finds the single most probable path, which is usually, though not always, the most probable transcript; Exercise 3 builds a case where they differ. Beam search keeps several candidate prefixes instead and can add a language model.

### ✍️ Your turn: `greedy_decode`

Write `greedy_decode(log_probs, blank=0)` that returns the transcript (a list of `int`) for a `(T, C)` array of log-probabilities. Hint: `np.argmax(..., axis=1)` and your `ctc_collapse`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def greedy_decode_reference(log_probs, blank=0):
    return ctc_collapse(np.argmax(log_probs, axis=1), blank)

In [ ]:
def greedy_decode(log_probs, blank=0):
    """Best-path decoding: the most likely label in every frame, then the collapse rule."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
def one_hot_path(path, n_labels=4, confidence=0.9):
    probs = np.full((len(path), n_labels), (1 - confidence) / (n_labels - 1))
    probs[np.arange(len(path)), path] = confidence
    return np.log(probs)


greedy_decode = practice("greedy_decode", greedy_decode, greedy_decode_reference, cases=[
    (one_hot_path([0, 1, 1, 0, 0, 2, 0]),),
    (one_hot_path([3, 0, 3, 3]),),
    (one_hot_path([0, 0, 0]),),
    (log_softmax_rows(np.random.default_rng(11).normal(size=(9, 4))),),
])

## 8 · Train a recogniser with your CTC loss

Now everything comes together. The data: 400 training, 60 validation and 100 test utterances, each generated with its own seed. The features: log-mel frames, normalised with means and standard deviations computed on the **training** frames only, with the two neighbouring frames on each side appended to every frame so that the model sees 80 ms of context ($5 \times 20 = 100$ inputs per frame).

The model: a dense layer of 32 ReLU units, then a dense layer to 4 outputs and `log_softmax`, applied to every frame. One detail matters for CTC: the output bias of the blank starts at $+2$, so the untrained model already says "blank" in most frames, which is where trained CTC models end up. Without it, training can get stuck in a solution that never uses the blank (see the common errors below). The optimiser is Adam, and every step averages your `ctc_op` over 16 utterances.

In [ ]:
def make_dataset(n, seed, noise=0.05, lengths=(2, 5)):
    """n utterances of 2 to 4 random symbols: their log-mel features and transcripts."""
    rng = np.random.default_rng(seed)
    features, targets = [], []
    for _ in range(n):
        labels = [int(k) for k in rng.integers(1, 4, size=int(rng.integers(*lengths)))]
        features.append(log_mel_spectrogram(make_utterance(labels, rng, noise), SAMPLE_RATE))
        targets.append(labels)
    return features, targets


train_feats, train_targets = make_dataset(400, seed=1)
val_feats, val_targets = make_dataset(60, seed=2)
test_feats, test_targets = make_dataset(100, seed=3)
frame_counts = [len(f) for f in train_feats]
print(f"training utterances: {len(train_feats)}, frames per utterance {min(frame_counts)} to {max(frame_counts)}")
print("examples:", [to_text(t) for t in train_targets[:6]])

feature_mean = np.concatenate(train_feats).mean(axis=0)          # training frames only
feature_std = np.concatenate(train_feats).std(axis=0) + 1e-8


def prepare(features, context=2):
    """Normalise with the training statistics and append `context` neighbouring frames on each side."""
    z = (features - feature_mean) / feature_std
    padded = np.pad(z, ((context, context), (0, 0)), mode="edge")
    return np.concatenate([padded[i:i + len(z)] for i in range(2 * context + 1)], axis=1)


train_inputs = [prepare(f) for f in train_feats]
print("one prepared utterance:", train_inputs[0].shape)

In [ ]:
model_rng = np.random.default_rng(0)
W1, b1 = parameter(100, 32, rng=model_rng), zeros_parameter(32)
W2, b2 = parameter(32, 4, rng=model_rng, scale=0.1), zeros_parameter(4)
b2.data[BLANK] = 2.0                  # start by predicting mostly blank, as trained CTC models do
params = [W1, b1, W2, b2]


def frame_log_probs(x):
    """(T, 100) prepared features -> (T, 4) log-probabilities over blank, a, b and c."""
    return ((Tensor(x) @ W1 + b1).relu() @ W2 + b2).log_softmax(axis=-1)


class Adam:
    """Adam on a list of engine parameters (as in Base Model Training)."""

    def __init__(self, params, learning_rate=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params, self.learning_rate, self.beta1, self.beta2, self.eps = params, learning_rate, beta1, beta2, eps
        self.m = [np.zeros_like(p.data) for p in params]
        self.v = [np.zeros_like(p.data) for p in params]
        self.t = 0

    def step(self):
        self.t += 1
        for p, m, v in zip(self.params, self.m, self.v):
            m[:] = self.beta1 * m + (1 - self.beta1) * p.grad
            v[:] = self.beta2 * v + (1 - self.beta2) * p.grad ** 2
            m_hat = m / (1 - self.beta1 ** self.t)
            v_hat = v / (1 - self.beta2 ** self.t)
            p.data -= self.learning_rate * m_hat / (np.sqrt(v_hat) + self.eps)


def edit_distance(reference, hypothesis):
    """Levenshtein distance: the fewest substitutions, deletions and insertions (you build this in notebook 04)."""
    previous = list(range(len(hypothesis) + 1))
    for i, ref_symbol in enumerate(reference, start=1):
        current = [i]
        for j, hyp_symbol in enumerate(hypothesis, start=1):
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + (ref_symbol != hyp_symbol)))
        previous = current
    return previous[-1]


def transcribe(features):
    return greedy_decode(frame_log_probs(prepare(features)).data)


def symbol_error_rate(features, targets):
    """Total edits divided by total reference symbols, over a whole set of utterances."""
    edits = sum(edit_distance(target, transcribe(f)) for f, target in zip(features, targets))
    return edits / sum(len(target) for target in targets)


print("before training, validation symbol error rate:", round(symbol_error_rate(val_feats, val_targets), 3))

In [ ]:
optimizer = Adam(params, learning_rate=0.01)
batch_rng = np.random.default_rng(5)
losses, checkpoints = [], []
start = time.perf_counter()
for step in range(1, 251):
    batch = batch_rng.choice(len(train_inputs), size=16, replace=False)
    loss = sum(ctc_op(frame_log_probs(train_inputs[i]), train_targets[i]) for i in batch) / len(batch)
    zero_grad(params)
    loss.backward()
    optimizer.step()
    losses.append(float(loss.data))
    if step % 50 == 0:
        checkpoints.append((step, symbol_error_rate(val_feats, val_targets)))
        print(f"step {step:3d}: mean CTC loss {np.mean(losses[-50:]):7.3f}, validation symbol error rate {checkpoints[-1][1]:.3f}")
print(f"training took {time.perf_counter() - start:.1f} s")

fig, axes = plt.subplots(1, 2, figsize=(9, 2.8))
axes[0].plot(losses); axes[0].set_yscale("log"); axes[0].set_xlabel("step"); axes[0].set_ylabel("CTC loss per utterance")
axes[1].plot(*zip(*checkpoints), "o-"); axes[1].set_xlabel("step"); axes[1].set_ylabel("validation symbol error rate")
plt.tight_layout(); plt.show()

**Expected behaviour.** The loss starts near 10 per utterance and falls by more than three orders of magnitude, and the validation symbol error rate drops to zero within about a hundred steps. Now evaluate once on the test utterances, which played no part in training or in watching the validation curve, and look at what the model outputs frame by frame.

In [ ]:
test_ser = symbol_error_rate(test_feats, test_targets)
print(f"held-out test symbol error rate: {test_ser:.3f} over {sum(len(t) for t in test_targets)} symbols")
for f, target in list(zip(test_feats, test_targets))[:5]:
    print(f"  reference {to_text(target):5s} -> recognised {to_text(transcribe(f))}")

posteriors = np.exp(frame_log_probs(prepare(test_feats[0])).data)
fig, axes = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
axes[0].imshow(test_feats[0].T, origin="lower", aspect="auto", cmap="magma"); axes[0].set_ylabel("mel band")
axes[0].set_title(f"Test utterance '{to_text(test_targets[0])}': features and per-frame probabilities")
for k in range(4):
    axes[1].plot(posteriors[:, k], label=ALPHABET[k])
axes[1].set_xlabel("frame"); axes[1].set_ylabel("probability"); axes[1].legend(ncol=4, fontsize=8)
plt.tight_layout(); plt.show()

**What to notice.** The blank dominates the pauses, and each tone is claimed by its own symbol for a run of frames. CTC models often become even "peakier", with a symbol in a single frame and blanks around it; either way, the collapse rule turns the frames into the transcript.

## 9 · 🧪 Experiments: where the recogniser breaks

**A shifted condition.** The model was trained with noise of standard deviation 0.05. Evaluate it on test sets with more noise. A low error rate under the training condition says nothing about a condition the model has never met.

In [ ]:
noise_levels = [0.05, 0.1, 0.2, 0.4]   # try adding 0.15, then run this cell again
for noise in noise_levels:
    shifted_feats, shifted_targets = make_dataset(100, seed=30, noise=noise)
    print(f"noise {noise:4.2f}: test symbol error rate {symbol_error_rate(shifted_feats, shifted_targets):.3f}")

**Expected behaviour.** No errors at the training noise level, a few at 0.1, and nearly every symbol lost at 0.2 and above. Louder noise raises the floor of every silent frame's features to values the model never saw, so it no longer finds the tones. The cure is training data that covers the conditions you care about (notebook 03 mixes noise in as augmentation), not a better score on the matched test set.

**Repeats without a pause.** CTC can only output "aa" if some frame between the two symbols is labelled blank. The next cell renders each repeated pair as one long, unbroken tone, which is the acoustic equivalent of saying a doubled letter as one sound, and counts how many repeats survive decoding.

In [ ]:
def make_utterance_joined(labels, rng, noise=0.05):
    """Like make_utterance, but consecutive equal symbols become one long tone with no pause."""
    groups = [[labels[0], 1]]
    for label in labels[1:]:
        if label == groups[-1][0]:
            groups[-1][1] += 1
        else:
            groups.append([label, 1])
    pieces = [np.zeros(int(rng.integers(256, 513)))]
    for label, count in groups:
        n = int(rng.integers(512, 897)) * count
        fade = np.minimum(1.0, np.minimum(np.arange(n), n - 1 - np.arange(n)) / 40)
        pieces += [fade * make_tone(TONE_HZ[label], n / SAMPLE_RATE, SAMPLE_RATE, 0.6), np.zeros(int(rng.integers(384, 641)))]
    x = np.concatenate(pieces)
    return x + noise * rng.normal(size=len(x))


joined_rng = np.random.default_rng(12)
targets_with_repeats = [[1, 1, 2], [3, 3], [2, 1, 1], [2, 2, 3, 3], [1, 2, 2]]
for target in targets_with_repeats:
    with_pause = transcribe(log_mel_spectrogram(make_utterance(target, joined_rng), SAMPLE_RATE))
    joined = transcribe(log_mel_spectrogram(make_utterance_joined(target, joined_rng), SAMPLE_RATE))
    print(f"reference {to_text(target):5s}: with pauses -> {to_text(with_pause):5s}  joined -> {to_text(joined)}")

**Expected behaviour.** With pauses the repeats are recognised; joined, a doubled symbol usually comes out once, because nothing in the sound separates the two copies. Real speech has the same problem with doubled sounds, and recognisers rely on context and language models to restore them.

## 10 · Put it together

The checks below restate what this notebook has shown, using your functions: the forward algorithm equals brute force, the gradient equals finite differences, impossible targets are reported as infinite, training reduced the loss, and the recogniser transcribes held-out utterances.

In [ ]:
final_rng = np.random.default_rng(99)
for T, target in [(5, [1, 2]), (6, [2, 2, 1]), (4, [])]:
    lp = log_softmax_rows(final_rng.normal(size=(T, 4)))
    assert np.isclose(-ctc_loss(lp, target), brute_force_log_prob(lp, target)), "forward algorithm != brute force"
assert gradient_error < 1e-6 and error_alone < 1e-6 and error_composed < 1e-6, "gradient checks should pass"
assert ctc_loss(log_softmax_rows(final_rng.normal(size=(2, 4))), [1, 1]) == np.inf, "'aa' cannot fit in 2 frames"
assert np.mean(losses[-25:]) < 0.05 * np.mean(losses[:25]), "training should cut the CTC loss by more than 95%"
assert test_ser <= 0.10, f"expected a held-out symbol error rate of at most 0.10, got {test_ser:.3f}"
print(f"✅ Your CTC recogniser: loss {np.mean(losses[:25]):.1f} -> {np.mean(losses[-25:]):.3f} per utterance, "
      f"held-out symbol error rate {test_ser:.3f}, trained without a single frame-level label.")

**Honest scope.** This is a 3,400-parameter model on three generated tones, so a near-zero error rate is expected and says nothing about speech. Production CTC recognisers use the same loss with deep convolutional, recurrent or transformer encoders, sub-word vocabularies of hundreds or thousands of units, thousands of hours of transcribed audio, and beam search with a language model. Frameworks such as PyTorch provide a CTC loss that takes `(T, batch, C)` log-probabilities plus input and target lengths, and computes exactly what your `ctc_loss_and_grad` computes, in compiled code.

## Exercises

### Exercise 1: `min_ctc_frames`

A target of length $L$ needs at least $L$ frames, plus one blank frame for every pair of equal neighbouring symbols. Write `min_ctc_frames(target)` that returns that minimum as an `int`. Then use it: if an encoder halves the frame rate, which of the training targets above could no longer fit in the shortest training utterance?

In [ ]:
def min_ctc_frames_reference(target):
    return len(target) + sum(1 for a, b in zip(target[:-1], target[1:]) if a == b)

In [ ]:
def min_ctc_frames(target):
    """The fewest frames that can carry this target under CTC."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
min_ctc_frames = practice("min_ctc_frames", min_ctc_frames, min_ctc_frames_reference, cases=[
    ([1, 2, 3],), ([1, 1],), ([2, 2, 2, 1],), ([],),
])

### Exercise 2: `ctc_viterbi`, forced alignment

Replace every `np.logaddexp` in the forward recursion by `np.maximum`, remember which of stay, step or skip won at each `(t, s)`, and trace back from the better of the two end positions. The result is the single most probable path that collapses to the target: a **forced alignment** that tells you which frames belong to which symbol. Write `ctc_viterbi(log_probs, target, blank=0)` returning that path as a list of `T` labels. (Ties are vanishingly unlikely with random scores; if you meet one, prefer stay, then step, then skip.)

In [ ]:
def ctc_viterbi_reference(log_probs, target, blank=0):
    ext = extend_with_blanks(target, blank)
    T, S = log_probs.shape[0], len(ext)
    score = np.full((T, S), -np.inf)
    back = np.zeros((T, S), dtype=int)
    score[0, 0] = log_probs[0, ext[0]]
    if S > 1:
        score[0, 1] = log_probs[0, ext[1]]
    can_skip = np.zeros(S, dtype=bool)
    can_skip[2:] = (ext[2:] != blank) & (ext[2:] != ext[:-2])
    for t in range(1, T):
        options = np.full((3, S), -np.inf)
        options[0] = score[t - 1]
        options[1, 1:] = score[t - 1, :-1]
        options[2, 2:] = np.where(can_skip[2:], score[t - 1, :-2], -np.inf)
        back[t] = np.argmax(options, axis=0)
        score[t] = options[back[t], np.arange(S)] + log_probs[t, ext]
    s = S - 1 if S == 1 or score[-1, -1] >= score[-1, -2] else S - 2
    path = [s]
    for t in range(T - 1, 0, -1):
        s -= back[t, s]
        path.append(s)
    return [int(ext[s]) for s in reversed(path)]

In [ ]:
def ctc_viterbi(log_probs, target, blank=0):
    """The most probable single path (T labels) that collapses to target."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
viterbi_rng = np.random.default_rng(13)
ctc_viterbi = practice("ctc_viterbi", ctc_viterbi, ctc_viterbi_reference, cases=[
    (log_softmax_rows(viterbi_rng.normal(size=(6, 4))), [1, 2]),
    (log_softmax_rows(viterbi_rng.normal(size=(7, 4))), [3, 3]),
    (log_softmax_rows(viterbi_rng.normal(size=(4, 4))), []),
])
print("forced alignment of the first test utterance:",
      to_text(ctc_viterbi(frame_log_probs(prepare(test_feats[0])).data, test_targets[0])))

### Exercise 3: the best path is not always the best transcript

Take two frames and two labels, blank and a, with $y_t(\text{blank}) = 0.6$ and $y_t(a) = 0.4$ in both frames. Greedy decoding returns the empty transcript, because the best single path `__` has probability $0.36$. Use `brute_force_log_prob` to compute $P(\text{"a"})$, which sums the paths `a_`, `_a` and `aa`, and explain why prefix beam search, which adds up the paths of each candidate prefix, would choose "a".

### Exercise 4: an open question

Real recognisers often downsample the frame rate (for example, by stacking pairs of frames) to save computation. Using `min_ctc_frames` and the frame counts printed in section 8, explain what would happen to the loss of the longest targets if this model stacked **four** frames together. What would you change in the data or the model?

## Common errors and debugging

- **The loss is `inf`.** A target needs more frames than the utterance has after any downsampling; check `min_ctc_frames` against the frame count.
- **The loss is `nan`.** Something computed `inf - inf`, usually from working with probabilities instead of log-probabilities, or from an `inf` loss reaching the optimiser. Stay in log space with `np.logaddexp` and skip impossible targets.
- **The forward algorithm disagrees with brute force by a little.** The skip rule is wrong: skipping is never allowed into a blank, nor between two equal symbols.
- **Greedy output merges real repeats or keeps duplicates.** The collapse steps are in the wrong order: merge repeats first, then remove blanks.
- **Training stalls with one symbol predicted everywhere and no blanks.** The model has reached a solution that never uses the blank, where the blank's gradient vanishes. Start the blank's output bias high, as in section 8, and check the learning rate.
- **Excellent validation scores that vanish on new recordings.** Normalisation statistics or model choices used the test data, or the test condition differs from the training condition (section 9).

## Summary

CTC lets a model learn from (audio, transcript) pairs without frame labels. A blank symbol and the collapse rule define which frame-level paths spell a transcript; the forward algorithm adds up all of them in $O(T \cdot S)$ steps in log space; and the gradient is "prediction minus occupancy", computed from forward and backward variables, where the backward pass is the forward pass run in reverse. Wrapped as an engine operation, your CTC loss trained a recogniser that transcribes held-out tone sequences, and the experiments showed its limits: shifted noise and missing pauses between repeats.

## Next

**03 · Encoder–decoder recognition and adaptation**: build an attention decoder trained with teacher forcing, compare its failures with CTC's, and then make a model generalise to new speakers with augmentation and speaker-level splits.